In [1]:

# Q4 Step 1: Initialize SQLite database and sample data

import sqlite3
import pandas as pd
from pathlib import Path

project_q4 = Path.home() / "question_04_month_over_month"
project_q4.mkdir(parents=True, exist_ok=True)

db_path_q4 = project_q4 / "monthly_revenue.db"

conn_q4 = sqlite3.connect(str(db_path_q4))

conn_q4.execute("""
CREATE TABLE IF NOT EXISTS monthly_revenue (
    region TEXT NOT NULL,
    month_end TEXT NOT NULL,
    revenue INTEGER NOT NULL,
    PRIMARY KEY (region, month_end)
)
""")

sample_rows_q4 = [
    ("South", "2026-01-31", 10000),
    ("South", "2026-02-28", 12000),
    ("South", "2026-03-31", 15000)
]

conn_q4.executemany("""
INSERT OR IGNORE INTO monthly_revenue
(region, month_end, revenue)
VALUES (?, ?, ?)
""", sample_rows_q4)

conn_q4.commit()

monthly_df_q4 = pd.read_sql_query("""
SELECT *
FROM monthly_revenue
ORDER BY region, month_end
""", conn_q4)

display(monthly_df_q4)

print("Database initialized successfully")


Database exists: True


,name
0,monthly_revenue


,region,month_end,revenue
0,South,2026-01-31,10000
1,South,2026-02-28,12000
2,South,2026-03-31,15000


Total records: 3


In [2]:

# Q4 Step 2: Previous revenue using SQL LAG()

lag_sql_q4 = """
SELECT
    region,
    month_end,
    revenue AS current_revenue,
    LAG(revenue, 1) OVER (
        PARTITION BY region
        ORDER BY month_end
    ) AS previous_revenue
FROM monthly_revenue
ORDER BY region, month_end
"""

lag_df_q4 = pd.read_sql_query(
    lag_sql_q4,
    conn_q4
)

display(lag_df_q4)

print("Total rows:", len(lag_df_q4))

print(
    "January previous revenue is missing:",
    pd.isna(lag_df_q4.loc[0, "previous_revenue"])
)


,region,month_end,current_revenue,previous_revenue
0,South,2026-01-31,10000,NaN
1,South,2026-02-28,12000,10000.0
2,South,2026-03-31,15000,12000.0


Total rows: 3
January previous revenue is missing: True


In [3]:

# Q4 Step 3: Calculate MoM absolute and percentage change

mom_sql_q4 = """
WITH revenue_with_lag AS (
    SELECT
        region,
        month_end,
        revenue AS current_revenue,

        LAG(revenue, 1) OVER (
            PARTITION BY region
            ORDER BY month_end
        ) AS previous_revenue

    FROM monthly_revenue
)
SELECT
    region,
    month_end,
    current_revenue,
    previous_revenue,

    current_revenue - previous_revenue
        AS mom_change,

    CASE
        WHEN previous_revenue IS NULL
             OR previous_revenue = 0
        THEN NULL

        ELSE ROUND(
            (current_revenue - previous_revenue)
            * 100.0 / previous_revenue,
            2
        )
    END AS mom_percentage

FROM revenue_with_lag
ORDER BY region, month_end
"""

mom_df_q4 = pd.read_sql_query(
    mom_sql_q4,
    conn_q4
)

display(mom_df_q4)

print("Total rows:", len(mom_df_q4))

print(
    "February MoM %:",
    mom_df_q4.loc[1, "mom_percentage"]
)

print(
    "March MoM %:",
    mom_df_q4.loc[2, "mom_percentage"]
)


,region,month_end,current_revenue,previous_revenue,mom_change,mom_percentage
0,South,2026-01-31,10000,NaN,NaN,NaN
1,South,2026-02-28,12000,10000.0,2000.0,20.0
2,South,2026-03-31,15000,12000.0,3000.0,25.0


Total rows: 3
February MoM %: 20.0
March MoM %: 25.0


In [4]:

# Q4 Step 4: Multiple regions using PARTITION BY

import pandas as pd

# Add North-region records safely
north_data_q4 = [
    ("North", "2026-01-31", 20000),
    ("North", "2026-02-28", 18000),
    ("North", "2026-03-31", 21000)
]

conn_q4.executemany("""
INSERT OR IGNORE INTO monthly_revenue
(region, month_end, revenue)
VALUES (?, ?, ?)
""", north_data_q4)

conn_q4.commit()

# Calculate MoM for each region independently
region_mom_sql_q4 = """
WITH revenue_with_lag AS (
    SELECT
        region,
        month_end,
        revenue AS current_revenue,

        LAG(revenue) OVER (
            PARTITION BY region
            ORDER BY month_end
        ) AS previous_revenue

    FROM monthly_revenue
)
SELECT
    region,
    month_end,
    current_revenue,
    previous_revenue,

    current_revenue - previous_revenue
        AS mom_change,

    CASE
        WHEN previous_revenue IS NULL
             OR previous_revenue = 0
        THEN NULL
        ELSE ROUND(
            (current_revenue - previous_revenue)
            * 100.0 / previous_revenue,
            2
        )
    END AS mom_percentage

FROM revenue_with_lag
ORDER BY region, month_end
"""

region_mom_df_q4 = pd.read_sql_query(
    region_mom_sql_q4,
    conn_q4
)

display(region_mom_df_q4)

print("Total records:", len(region_mom_df_q4))

print(
    "North February MoM:",
    region_mom_df_q4.loc[
        (region_mom_df_q4["region"] == "North")
        & (region_mom_df_q4["month_end"] == "2026-02-28"),
        "mom_percentage"
    ].iloc[0]
)


,region,month_end,current_revenue,previous_revenue,mom_change,mom_percentage
0,North,2026-01-31,20000,NaN,NaN,NaN
1,North,2026-02-28,18000,20000.0,-2000.0,-10.00
2,North,2026-03-31,21000,18000.0,3000.0,16.67
3,South,2026-01-31,10000,NaN,NaN,NaN
4,South,2026-02-28,12000,10000.0,2000.0,20.00
5,South,2026-03-31,15000,12000.0,3000.0,25.00


Total records: 6
North February MoM: -10.0


In [6]:

# Q4 Step 5: Demonstrate missing-month problem

missing_month_sql_q4 = """
WITH missing_month_example AS (
    SELECT
        'South' AS region,
        '2026-01-31' AS month_end,
        10000 AS revenue

    UNION ALL

    SELECT
        'South',
        '2026-03-31',
        15000
),
revenue_with_lag AS (
    SELECT
        region,
        month_end,
        revenue,

        LAG(month_end) OVER (
            PARTITION BY region
            ORDER BY month_end
        ) AS previous_available_month,

        LAG(revenue) OVER (
            PARTITION BY region
            ORDER BY month_end
        ) AS previous_available_revenue

    FROM missing_month_example
)
SELECT
    region,
    month_end,
    revenue,
    previous_available_month,
    previous_available_revenue,

    CASE
        WHEN previous_available_revenue IS NULL
             OR previous_available_revenue = 0
        THEN NULL

        ELSE ROUND(
            (revenue - previous_available_revenue)
            * 100.0 / previous_available_revenue,
            2
        )
    END AS naive_growth_percentage

FROM revenue_with_lag
ORDER BY month_end
"""

missing_month_df_q4 = pd.read_sql_query(
    missing_month_sql_q4,
    conn_q4
)

display(missing_month_df_q4)

print("Rows returned:", len(missing_month_df_q4))

print(
    "March previous available month:",
    missing_month_df_q4.loc[
        1, "previous_available_month"
    ]
)

print(
    "March naive growth:",
    missing_month_df_q4.loc[
        1, "naive_growth_percentage"
    ]
)


,region,month_end,revenue,previous_available_month,previous_available_revenue,naive_growth_percentage
0,South,2026-01-31,10000,None,NaN,NaN
1,South,2026-03-31,15000,2026-01-31,10000.0,50.0


Rows returned: 2
March previous available month: 2026-01-31
March naive growth: 50.0


In [7]:

# Q4 Step 6: Date spine + missing-month handling

import pandas as pd

date_spine_sql_q4 = """
WITH RECURSIVE month_spine(month_start) AS (
    SELECT DATE('2026-01-01')

    UNION ALL

    SELECT DATE(month_start, '+1 month')
    FROM month_spine
    WHERE month_start < '2026-03-01'
),

source_revenue AS (
    SELECT
        'South' AS region,
        '2026-01-31' AS month_end,
        10000 AS revenue

    UNION ALL

    SELECT
        'South',
        '2026-03-31',
        15000
),

calendar_revenue AS (
    SELECT
        'South' AS region,
        m.month_start,
        s.revenue
    FROM month_spine AS m
    LEFT JOIN source_revenue AS s
        ON DATE(s.month_end, 'start of month')
           = m.month_start
),

revenue_with_lag AS (
    SELECT
        region,
        month_start,
        revenue,

        LAG(revenue) OVER (
            PARTITION BY region
            ORDER BY month_start
        ) AS previous_revenue

    FROM calendar_revenue
)

SELECT
    region,
    month_start,
    revenue,
    previous_revenue,

    CASE
        WHEN previous_revenue IS NULL
             OR previous_revenue = 0
             OR revenue IS NULL
        THEN NULL

        ELSE ROUND(
            (revenue - previous_revenue)
            * 100.0 / previous_revenue,
            2
        )
    END AS mom_percentage

FROM revenue_with_lag
ORDER BY region, month_start
"""

date_spine_df_q4 = pd.read_sql_query(
    date_spine_sql_q4,
    conn_q4
)

display(date_spine_df_q4)

print("Calendar months:", len(date_spine_df_q4))

print(
    "February revenue missing:",
    pd.isna(date_spine_df_q4.loc[1, "revenue"])
)

print(
    "March MoM correctly undefined:",
    pd.isna(date_spine_df_q4.loc[2, "mom_percentage"])
)


,region,month_start,revenue,previous_revenue,mom_percentage
0,South,2026-01-01,10000.0,NaN,None
1,South,2026-02-01,NaN,10000.0,None
2,South,2026-03-01,15000.0,NaN,None


Calendar months: 3
February revenue missing: True
March MoM correctly undefined: True


In [8]:

# Q4 Step 7: Source-to-target reconciliation
# Reuses Step 6 date_spine_df_q4

import pandas as pd

# Define the same source records used in Step 6
source_sql_q4_s7 = """
SELECT
    'South' AS region,
    '2026-01-01' AS month_start,
    10000 AS revenue

UNION ALL

SELECT
    'South',
    '2026-03-01',
    15000
"""

# Calculate source metrics using SQL
source_metrics_sql_q4_s7 = """
WITH source_revenue AS (
    SELECT 'South' AS region,
           '2026-01-01' AS month_start,
           10000 AS revenue
    UNION ALL
    SELECT 'South', '2026-03-01', 15000
)
SELECT
    COUNT(*) AS source_rows,
    COALESCE(SUM(revenue), 0) AS source_total
FROM source_revenue
"""

source_metrics_df_q4_s7 = pd.read_sql_query(
    source_metrics_sql_q4_s7,
    conn_q4
)

# Calculate target metrics from Step 6 result
target_rows_q4_s7 = len(date_spine_df_q4)

target_non_null_q4_s7 = (
    date_spine_df_q4["revenue"].count()
)

missing_months_q4_s7 = (
    date_spine_df_q4["revenue"].isna().sum()
)

target_total_q4_s7 = (
    date_spine_df_q4["revenue"].sum()
)

unique_months_q4_s7 = (
    date_spine_df_q4["month_start"].nunique()
)

# Extract source metrics
source_rows_q4_s7 = int(
    source_metrics_df_q4_s7.loc[0, "source_rows"]
)

source_total_q4_s7 = int(
    source_metrics_df_q4_s7.loc[0, "source_total"]
)

# Final validation
recon_pass_q4_s7 = (
    source_rows_q4_s7 == target_non_null_q4_s7
    and source_total_q4_s7 == target_total_q4_s7
    and target_rows_q4_s7 == 3
    and unique_months_q4_s7 == 3
    and missing_months_q4_s7 == 1
    and pd.isna(
        date_spine_df_q4.loc[2, "mom_percentage"]
    )
)

print("Source rows:", source_rows_q4_s7)
print("Target non-null revenue rows:", target_non_null_q4_s7)

print("Source revenue total:", source_total_q4_s7)
print("Target revenue total:", target_total_q4_s7)

print("Calendar rows:", target_rows_q4_s7)
print("Unique calendar months:", unique_months_q4_s7)
print("Missing revenue months:", missing_months_q4_s7)

print(
    "Q4 STEP 7 RECONCILIATION:",
    "PASS" if recon_pass_q4_s7 else "FAIL"
)


Source rows: 2
Target non-null revenue rows: 2
Source revenue total: 25000
Target revenue total: 25000.0
Calendar rows: 3
Unique calendar months: 3
Missing revenue months: 1
Q4 STEP 7 RECONCILIATION: PASS


In [9]:

# Q4 Step 8: Record-level reconciliation

import pandas as pd

# Source records from our missing-month example
source_df_q4_s8 = pd.read_sql_query("""
SELECT
    'South' AS region,
    '2026-01-01' AS month_start,
    10000 AS revenue

UNION ALL

SELECT
    'South',
    '2026-03-01',
    15000
""", conn_q4)

# Our date spine result from Step 6
target_df_q4_s8 = date_spine_df_q4[
    ["region", "month_start", "revenue"]
].copy()

# Keep the actual populated target records
target_populated_q4_s8 = target_df_q4_s8[
    target_df_q4_s8["revenue"].notna()
].copy()

# Convert revenue to integer for consistent comparison
target_populated_q4_s8["revenue"] = (
    target_populated_q4_s8["revenue"].astype(int)
)

# Load the target into a temporary SQLite table
target_populated_q4_s8.to_sql(
    "q4_step8_target",
    conn_q4,
    if_exists="replace",
    index=False
)

# Compare exact records in both directions
except_sql_q4_s8 = """
WITH source_revenue AS (
    SELECT 'South' AS region,
           '2026-01-01' AS month_start,
           10000 AS revenue
    UNION ALL
    SELECT 'South', '2026-03-01', 15000
),
source_minus_target AS (
    SELECT region, month_start, revenue
    FROM source_revenue

    EXCEPT

    SELECT region, month_start, revenue
    FROM q4_step8_target
),
target_minus_source AS (
    SELECT region, month_start, revenue
    FROM q4_step8_target

    EXCEPT

    SELECT region, month_start, revenue
    FROM source_revenue
)
SELECT
    (SELECT COUNT(*) FROM source_minus_target)
        AS source_minus_target,
    (SELECT COUNT(*) FROM target_minus_source)
        AS target_minus_source
"""

except_df_q4_s8 = pd.read_sql_query(
    except_sql_q4_s8,
    conn_q4
)

display(except_df_q4_s8)

# Validate February as a missing source month
february_q4_s8 = target_df_q4_s8[
    target_df_q4_s8["month_start"] == "2026-02-01"
]

february_missing_q4_s8 = (
    len(february_q4_s8) == 1
    and february_q4_s8["revenue"].isna().all()
)

# Validate the record-level reconciliation
source_diff_q4_s8 = int(
    except_df_q4_s8.loc[0, "source_minus_target"]
)

target_diff_q4_s8 = int(
    except_df_q4_s8.loc[0, "target_minus_source"]
)

duplicate_months_q4_s8 = int(
    target_df_q4_s8.duplicated(
        subset=["region", "month_start"]
    ).sum()
)

recon_pass_q4_s8 = (
    source_diff_q4_s8 == 0
    and target_diff_q4_s8 == 0
    and len(target_df_q4_s8) == 3
    and duplicate_months_q4_s8 == 0
    and february_missing_q4_s8
    and pd.isna(
        date_spine_df_q4.loc[2, "mom_percentage"]
    )
)

print("Source minus target:", source_diff_q4_s8)
print("Target minus source:", target_diff_q4_s8)
print("Duplicate region-month keys:", duplicate_months_q4_s8)
print("February correctly missing:", february_missing_q4_s8)

print(
    "Q4 STEP 8 RECORD RECONCILIATION:",
    "PASS" if recon_pass_q4_s8 else "FAIL"
)


,source_minus_target,target_minus_source
0,0,0


Source minus target: 0
Target minus source: 0
Duplicate region-month keys: 0
February correctly missing: True
Q4 STEP 8 RECORD RECONCILIATION: PASS


In [10]:

# Q4 Step 9: Final project validation
# Reuses verified results from Steps 4, 6 and 8

import pandas as pd

# 1. Validate the multi-region revenue calculation
def get_mom_q4(region, month_end):
    matches = region_mom_df_q4.loc[
        (region_mom_df_q4["region"] == region)
        & (region_mom_df_q4["month_end"] == month_end),
        "mom_percentage"
    ]

    assert len(matches) == 1, (
        "Expected one row for {} {}".format(
            region, month_end
        )
    )

    return matches.iloc[0]


assert abs(
    get_mom_q4("South", "2026-02-28") - 20.0
) < 0.001, "South February MoM mismatch"

assert abs(
    get_mom_q4("South", "2026-03-31") - 25.0
) < 0.001, "South March MoM mismatch"

assert abs(
    get_mom_q4("North", "2026-02-28") - (-10.0)
) < 0.001, "North February MoM mismatch"

print("Multi-region MoM validation: PASS")

# 2. Validate the date-spine scenario
assert len(date_spine_df_q4) == 3, (
    "Incorrect calendar row count"
)

feb_q4_s9 = date_spine_df_q4.loc[
    date_spine_df_q4["month_start"] == "2026-02-01",
    "revenue"
]

assert len(feb_q4_s9) == 1, (
    "February calendar row missing or duplicated"
)

assert pd.isna(feb_q4_s9.iloc[0]), (
    "February revenue should be missing"
)

march_q4_s9 = date_spine_df_q4.loc[
    date_spine_df_q4["month_start"] == "2026-03-01",
    "mom_percentage"
]

assert len(march_q4_s9) == 1, (
    "March calendar row missing or duplicated"
)

assert pd.isna(march_q4_s9.iloc[0]), (
    "March MoM must be undefined"
)

print("Missing-month validation: PASS")

# 3. Validate exact source-target reconciliation
assert source_diff_q4_s8 == 0, (
    "Source minus target is not zero"
)

assert target_diff_q4_s8 == 0, (
    "Target minus source is not zero"
)

assert duplicate_months_q4_s8 == 0, (
    "Duplicate region-month keys found"
)

print("Record-level reconciliation: PASS")

# 4. Final success
print("Q4 FINAL VALIDATION: PASS")


Multi-region MoM validation: PASS
Missing-month validation: PASS
Record-level reconciliation: PASS
Q4 FINAL VALIDATION: PASS


In [11]:

# Q4 Step 10A: Verify project files safely

from pathlib import Path

project_q4 = Path.home() / "question_04_month_over_month"

print("PROJECT FILES:")

for file in sorted(project_q4.iterdir()):
    print(file.name)

print("\nVALIDATION:")

notebook_exists_q4 = (
    project_q4 / "SQL_Question_04_Month_Over_Month.ipynb"
).is_file()

database_exists_q4 = (
    project_q4 / "monthly_revenue.db"
).is_file()

print("Notebook exists:", notebook_exists_q4)
print("Database exists:", database_exists_q4)

print(
    "STEP 10A:",
    "PASS" if notebook_exists_q4 and database_exists_q4
    else "CHECK REQUIRED"
)


PROJECT FILES:
.ipynb_checkpoints
SQL_Question_04_Month_Over_Month.ipynb
monthly_revenue.db

VALIDATION:
Notebook exists: True
Database exists: True
STEP 10A: PASS


In [12]:

# Q4 Step 10B: Create GitHub README.md

from pathlib import Path

project_q4 = Path.home() / "question_04_month_over_month"

readme_q4 = """# SQL Question 4 — Month-over-Month Revenue Analysis

## Business Problem

A banking revenue reporting pipeline needs to calculate
month-over-month (MoM) growth across multiple regions.

A key challenge is handling missing calendar months,
because SQL LAG() retrieves the previous available row,
not necessarily the previous calendar month.

## Objectives

- Calculate previous-month revenue using LAG().
- Calculate MoM absolute and percentage change.
- Perform region-wise calculations using PARTITION BY.
- Detect missing reporting periods.
- Generate a monthly date spine.
- Avoid misleading MoM growth when revenue is missing.
- Validate source and target records.

## Technologies

- SQL (SQLite)
- Python
- pandas
- JupyterLab

## SQL Concepts Practiced

- LAG() window function
- OVER(), PARTITION BY, ORDER BY
- Common Table Expressions (CTEs)
- WITH RECURSIVE
- UNION ALL
- LEFT JOIN
- CASE WHEN
- COALESCE
- COUNT, SUM and DISTINCT
- Bidirectional EXCEPT reconciliation

## Example 1 — Normal MoM Calculation

| Region | Month | Revenue | MoM Growth |
|---|---|---:|---:|
| South | January | 10000 | NULL |
| South | February | 12000 | +20% |
| South | March | 15000 | +25% |
| North | January | 20000 | NULL |
| North | February | 18000 | -10% |
| North | March | 21000 | +16.67% |

## Example 2 — Missing-Month Scenario

| Month | Revenue | MoM Growth |
|---|---:|---:|
| January | 10000 | NULL |
| February | NULL | NULL |
| March | 15000 | NULL |

Without calendar validation, a naive LAG() calculation
would compare March against January and report 50%
growth, which is not a valid February-to-March comparison.

## Validation Results

- Normal regional MoM calculations: PASS
- Missing-month detection: PASS
- March MoM correctly undefined: PASS
- Source-to-target record differences: 0
- Duplicate region-month keys: 0
- Final notebook validation: PASS

## How to Run

1. Install Python, pandas and JupyterLab.
2. Open SQL_Question_04_Month_Over_Month.ipynb.
3. Run all cells in sequence.
4. Verify the final validation output.

The notebook includes code to create sample SQLite tables.
The local monthly_revenue.db file is not required for upload.

## Key Learning

A successful SQL window function can still produce
incorrect business results when reporting periods are missing.

Always validate calendar completeness and business grain
before publishing financial MoM metrics.
"""

readme_path_q4 = project_q4 / "README.md"

readme_path_q4.write_text(
    readme_q4,
    encoding="utf-8"
)

print("README created:", readme_path_q4)
print("README exists:", readme_path_q4.is_file())
print("README size:", readme_path_q4.stat().st_size, "bytes")


README created: /home/itv028683/question_04_month_over_month/README.md
README exists: True
README size: 2405 bytes


In [15]:

# Q4 Step 10C: Inspect notebook reproducibility

import json
from pathlib import Path

# 1. Locate our existing notebook
project_q4 = Path.home() / "question_04_month_over_month"

notebook_path_q4 = (
    project_q4 / "SQL_Question_04_Month_Over_Month.ipynb"
)

# 2. Read the notebook as JSON
with notebook_path_q4.open("r", encoding="utf-8") as file:
    notebook_data_q4 = json.load(file)

# 3. Extract all code cells
code_cells_q4 = [
    "".join(cell.get("source", []))
    for cell in notebook_data_q4.get("cells", [])
    if cell.get("cell_type") == "code"
]

all_code_q4 = "\n".join(code_cells_q4)

# 4. Check for important project components
checks_q4 = {
    "SQLite connection": "sqlite3.connect" in all_code_q4,

    "Creates revenue table": (
        "CREATE TABLE IF NOT EXISTS monthly_revenue"
        in all_code_q4
    ),

    "Inserts sample data": (
        "INSERT OR IGNORE INTO monthly_revenue"
        in all_code_q4
    ),

    "LAG window function": (
        "LAG(" in all_code_q4
    ),

    "Regional partitioning": (
        "PARTITION BY region" in all_code_q4
    ),

    "Recursive date spine": (
        "WITH RECURSIVE" in all_code_q4
    ),

    "Record reconciliation": (
        "EXCEPT" in all_code_q4
    ),

    "Final validation": (
        "Q4 FINAL VALIDATION: PASS" in all_code_q4
    )
}

# 5. Display individual check results
for check_name, passed in checks_q4.items():
    print(
        "{}: {}".format(
            check_name,
            "PASS" if passed else "CHECK REQUIRED"
        )
    )

# 6. Overall content check
print("\nNotebook code cells:", len(code_cells_q4))

print(
    "Q4 REPRODUCIBILITY CONTENT CHECK:",
    "PASS" if all(checks_q4.values())
    else "REVIEW REQUIRED"
)


SQLite connection: PASS
Creates revenue table: PASS
Inserts sample data: PASS
LAG window function: PASS
Regional partitioning: PASS
Recursive date spine: PASS
Record reconciliation: PASS
Final validation: PASS

Notebook code cells: 13
Q4 REPRODUCIBILITY CONTENT CHECK: PASS


In [16]:

# Q4 Step 10D.1: Prepare isolated reproducibility test

from pathlib import Path
import json
import shutil

# Original project
project_q4 = (
    Path.home() / "question_04_month_over_month"
)

source_notebook_q4 = (
    project_q4
    / "SQL_Question_04_Month_Over_Month.ipynb"
)

# Separate test directory
test_folder_q4 = (
    Path.home() / "question_04_clean_test"
)

test_folder_q4.mkdir(
    parents=True,
    exist_ok=True
)

test_notebook_q4 = (
    test_folder_q4
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read the original notebook
with source_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_json_q4 = json.load(f)

# Redirect the known original project path
# in copied code cells only
old_path_q4 = (
    'Path.home() / "question_04_month_over_month"'
)

new_path_q4 = (
    'Path.home() / "question_04_clean_test"'
)

replacement_count_q4 = 0

for cell in notebook_json_q4.get("cells", []):
    if cell.get("cell_type") == "code":
        source = "".join(cell.get("source", []))

        replacement_count_q4 += source.count(
            old_path_q4
        )

        cell["source"] = source.replace(
            old_path_q4,
            new_path_q4
        ).splitlines(True)

# Save the isolated test notebook
with test_notebook_q4.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        notebook_json_q4,
        f,
        indent=1,
        ensure_ascii=False
    )

print("Original notebook:", source_notebook_q4)
print("Test notebook:", test_notebook_q4)

print(
    "Project-path replacements:",
    replacement_count_q4
)

print(
    "Test notebook exists:",
    test_notebook_q4.is_file()
)

print(
    "Original database untouched:",
    (project_q4 / "monthly_revenue.db").is_file()
)


Original notebook: /home/itv028683/question_04_month_over_month/SQL_Question_04_Month_Over_Month.ipynb
Test notebook: /home/itv028683/question_04_clean_test/SQL_Question_04_Month_Over_Month_TEST.ipynb
Project-path replacements: 5
Test notebook exists: True
Original database untouched: True


In [17]:

# Q4 Step 10D.2: Verify clean-test path isolation

import json
from pathlib import Path

test_folder_q4 = (
    Path.home() / "question_04_clean_test"
)

test_notebook_q4 = (
    test_folder_q4
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read copied notebook
with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    test_data_q4 = json.load(f)

# Combine executable code
test_code_q4 = "\n".join(
    "".join(cell.get("source", []))
    for cell in test_data_q4.get("cells", [])
    if cell.get("cell_type") == "code"
)

# Inspect path references
old_project_name_q4 = "question_04_month_over_month"
new_project_name_q4 = "question_04_clean_test"

old_path_count_q4 = test_code_q4.count(
    old_project_name_q4
)

new_path_count_q4 = test_code_q4.count(
    new_project_name_q4
)

# Check whether test database already exists
test_db_q4 = test_folder_q4 / "monthly_revenue.db"

print("Old project references:", old_path_count_q4)
print("Clean-test references:", new_path_count_q4)

print("Test notebook exists:", test_notebook_q4.is_file())
print("Test database already exists:", test_db_q4.exists())

print(
    "Path isolation:",
    "PASS" if (
        old_path_count_q4 == 0
        and new_path_count_q4 > 0
        and not test_db_q4.exists()
    ) else "REVIEW REQUIRED"
)


Old project references: 1
Clean-test references: 7
Test notebook exists: True
Test database already exists: False
Path isolation: REVIEW REQUIRED


In [18]:

# Q4 Step 10D.2A: Locate remaining original references

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_data_q4 = json.load(f)

search_text_q4 = "question_04_month_over_month"

print("REFERENCES TO ORIGINAL PROJECT:\n")

match_count_q4 = 0

for cell_number, cell in enumerate(
    notebook_data_q4.get("cells", []),
    start=1
):
    if cell.get("cell_type") != "code":
        continue

    cell_source = "".join(cell.get("source", []))

    for line_number, line in enumerate(
        cell_source.splitlines(),
        start=1
    ):
        if search_text_q4 in line:
            match_count_q4 += 1

            print("Cell number:", cell_number)
            print("Line number:", line_number)
            print("Code:", line.strip())
            print("-" * 40)

print("Total matching lines:", match_count_q4)


REFERENCES TO ORIGINAL PROJECT:

Cell number: 1
Line number: 11
Code: / "question_04_month_over_month"
----------------------------------------
Total matching lines: 1


In [19]:

# Q4 Step 10D.2B: Fix remaining test path

import json
from pathlib import Path

original_q4 = (
    Path.home()
    / "question_04_month_over_month"
)

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read isolated notebook
with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    test_data_q4 = json.load(f)

old_reference = '/ "question_04_month_over_month"'
new_reference = '/ "question_04_clean_test"'

replacements_q4 = 0

# Modify only the copied notebook's first code cell
first_cell_q4 = test_data_q4["cells"][0]

if first_cell_q4["cell_type"] != "code":
    raise ValueError("First notebook cell is not code")

source_q4 = "".join(first_cell_q4["source"])

replacements_q4 = source_q4.count(old_reference)

if replacements_q4 != 1:
    raise ValueError(
        "Expected exactly one original-path reference"
    )

first_cell_q4["source"] = source_q4.replace(
    old_reference,
    new_reference
).splitlines(True)

# Save the corrected test notebook
with test_notebook_q4.open(
    "w", encoding="utf-8"
) as f:
    json.dump(test_data_q4, f, indent=1)

# Verify remaining original-path references
all_code_q4 = "\n".join(
    "".join(cell.get("source", []))
    for cell in test_data_q4["cells"]
    if cell.get("cell_type") == "code"
)

remaining_q4 = all_code_q4.count(
    "question_04_month_over_month"
)

print("Path replacements:", replacements_q4)
print("Remaining original references:", remaining_q4)

print(
    "Original database exists:",
    (original_q4 / "monthly_revenue.db").is_file()
)

print(
    "Test database exists:",
    (
        Path.home()
        / "question_04_clean_test"
        / "monthly_revenue.db"
    ).exists()
)

print(
    "PATH CORRECTION:",
    "PASS" if replacements_q4 == 1
    and remaining_q4 == 0
    else "REVIEW REQUIRED"
)


Path replacements: 1
Remaining original references: 0
Original database exists: True
Test database exists: False
PATH CORRECTION: PASS


In [21]:

# Q4 Step 10D.3: Inspect executable cell boundaries

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read copied notebook
with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_data_q4 = json.load(f)

print("CODE CELL INVENTORY")
print("-" * 65)

code_count_q4 = 0

# Inspect each code cell without executing it
for cell_number, cell in enumerate(
    notebook_data_q4.get("cells", []),
    start=1
):
    if cell.get("cell_type") != "code":
        continue

    code_count_q4 += 1

    source_q4 = "".join(cell.get("source", []))

    # Find the first non-empty code line
    lines_q4 = [
        line.strip()
        for line in source_q4.splitlines()
        if line.strip()
    ]

    preview_q4 = (
        lines_q4[0][:90]
        if lines_q4
        else "(empty cell)"
    )

    print(
        "Cell {} | {}".format(
            cell_number,
            preview_q4
        )
    )

print("-" * 65)
print("Total code cells:", code_count_q4)
print("STEP 10D.3 INVENTORY: COMPLETE")


CODE CELL INVENTORY
-----------------------------------------------------------------
Cell 1 | # Q4 - Verify the existing SQLite database
Cell 2 | # Q4 Step 2: Previous revenue using SQL LAG()
Cell 3 | # Q4 Step 3: Calculate MoM absolute and percentage change
Cell 4 | # Q4 Step 4: Multiple regions using PARTITION BY
Cell 5 | # Q4 Step 5: Demonstrate missing-month problem
Cell 6 | # Q4 Step 6: Date spine + missing-month handling
Cell 7 | # Q4 Step 7: Source-to-target reconciliation
Cell 8 | # Q4 Step 8: Record-level reconciliation
Cell 9 | # Q4 Step 9: Final project validation
Cell 10 | # Q4 Step 10A: Verify project files safely
Cell 11 | # Q4 Step 10B: Create GitHub README.md
Cell 12 | # Q4 Step 10C: Inspect notebook reproducibility
Cell 13 | # Q4 Step 10D.1: Prepare isolated reproducibility test
-----------------------------------------------------------------
Total code cells: 13
STEP 10D.3 INVENTORY: COMPLETE


In [22]:

# Q4 Step 10D.3: Inspect executable cell boundaries

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read copied notebook
with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_data_q4 = json.load(f)

print("CODE CELL INVENTORY")
print("-" * 65)

code_count_q4 = 0

# Inspect each code cell without executing it
for cell_number, cell in enumerate(
    notebook_data_q4.get("cells", []),
    start=1
):
    if cell.get("cell_type") != "code":
        continue

    code_count_q4 += 1

    source_q4 = "".join(cell.get("source", []))

    # Find the first non-empty code line
    lines_q4 = [
        line.strip()
        for line in source_q4.splitlines()
        if line.strip()
    ]

    preview_q4 = (
        lines_q4[0][:90]
        if lines_q4
        else "(empty cell)"
    )

    print(
        "Cell {} | {}".format(
            cell_number,
            preview_q4
        )
    )

print("-" * 65)
print("Total code cells:", code_count_q4)
print("STEP 10D.3 INVENTORY: COMPLETE")


CODE CELL INVENTORY
-----------------------------------------------------------------
Cell 1 | # Q4 - Verify the existing SQLite database
Cell 2 | # Q4 Step 2: Previous revenue using SQL LAG()
Cell 3 | # Q4 Step 3: Calculate MoM absolute and percentage change
Cell 4 | # Q4 Step 4: Multiple regions using PARTITION BY
Cell 5 | # Q4 Step 5: Demonstrate missing-month problem
Cell 6 | # Q4 Step 6: Date spine + missing-month handling
Cell 7 | # Q4 Step 7: Source-to-target reconciliation
Cell 8 | # Q4 Step 8: Record-level reconciliation
Cell 9 | # Q4 Step 9: Final project validation
Cell 10 | # Q4 Step 10A: Verify project files safely
Cell 11 | # Q4 Step 10B: Create GitHub README.md
Cell 12 | # Q4 Step 10C: Inspect notebook reproducibility
Cell 13 | # Q4 Step 10D.1: Prepare isolated reproducibility test
-----------------------------------------------------------------
Total code cells: 13
STEP 10D.3 INVENTORY: COMPLETE


In [23]:

# Q4 Step 10D.4: Inspect Cell 1 database setup

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_data_q4 = json.load(f)

# Retrieve the first notebook cell
first_cell_q4 = notebook_data_q4["cells"][0]

# Convert its source lines into a single string
first_cell_code_q4 = "".join(
    first_cell_q4.get("source", [])
)

print("CELL 1 — DATABASE INITIALIZATION")
print("-" * 65)

print(first_cell_code_q4)

print("-" * 65)

# Inspect whether important SQL setup exists in Cell 1
checks_q4_s10d4 = {
    "SQLite connection": (
        "sqlite3.connect" in first_cell_code_q4
    ),
    "Table creation": (
        "CREATE TABLE" in first_cell_code_q4.upper()
    ),
    "Sample data insertion": (
        "INSERT" in first_cell_code_q4.upper()
    ),
    "Original project reference": (
        "question_04_month_over_month"
        in first_cell_code_q4
    )
}

print("\nINITIALIZATION CHECKS")

for name, result in checks_q4_s10d4.items():
    print("{}: {}".format(name, result))

print("\nSTEP 10D.4 INSPECTION: COMPLETE")


CELL 1 — DATABASE INITIALIZATION
-----------------------------------------------------------------

# Q4 - Verify the existing SQLite database

import sqlite3
import pandas as pd
from pathlib import Path

# Locate our database
db_path_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "monthly_revenue.db"
)

# Check that database file exists
print("Database exists:", db_path_q4.is_file())

# Connect to the existing database
if not db_path_q4.is_file():
    raise FileNotFoundError(db_path_q4)

conn_q4 = sqlite3.connect(str(db_path_q4))

# Inspect available tables
tables_q4 = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn_q4)

display(tables_q4)

# Display monthly revenue records
monthly_df_q4 = pd.read_sql_query("""
    SELECT *
    FROM monthly_revenue
    ORDER BY region, month_end
""", conn_q4)

display(monthly_df_q4)

print("Total records:", len(monthly_df_q4))

----------------------------------------------

In [24]:

# Q4 Step 10D.5: Fix initialization in TEST notebook only

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read test notebook
with test_notebook_q4.open("r", encoding="utf-8") as f:
    notebook_data_q4 = json.load(f)

# Replacement initialization code
setup_code_q4 = '''
# Q4 Step 1: Create SQLite database and sample data

import sqlite3
import pandas as pd
from pathlib import Path

db_path_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "monthly_revenue.db"
)

# Connect: SQLite creates database if missing
conn_q4 = sqlite3.connect(str(db_path_q4))

# Create table
conn_q4.execute("""
CREATE TABLE IF NOT EXISTS monthly_revenue (
    region TEXT NOT NULL,
    month_end TEXT NOT NULL,
    revenue INTEGER NOT NULL,
    PRIMARY KEY (region, month_end)
)
""")

# Initial South-region sample data
sample_rows_q4 = [
    ("South", "2026-01-31", 10000),
    ("South", "2026-02-28", 12000),
    ("South", "2026-03-31", 15000)
]

conn_q4.executemany("""
INSERT OR IGNORE INTO monthly_revenue
(region, month_end, revenue)
VALUES (?, ?, ?)
""", sample_rows_q4)

conn_q4.commit()

monthly_df_q4 = pd.read_sql_query("""
SELECT region, month_end, revenue
FROM monthly_revenue
ORDER BY region, month_end
""", conn_q4)

display(monthly_df_q4)

print("Database:", db_path_q4)
print("Initial records:", len(monthly_df_q4))
'''

# Modify the first code cell
first_cell_q4 = notebook_data_q4["cells"][0]

assert first_cell_q4["cell_type"] == "code"

first_cell_q4["source"] = (
    setup_code_q4.strip().splitlines(True)
)

# Save updated TEST notebook
with test_notebook_q4.open("w", encoding="utf-8") as f:
    json.dump(
        notebook_data_q4,
        f,
        indent=1,
        ensure_ascii=False
    )

# Verify changes
updated_cell_q4 = "".join(
    notebook_data_q4["cells"][0]["source"]
)

print("Table creation included:",
      "CREATE TABLE" in updated_cell_q4)

print("Data insertion included:",
      "INSERT OR IGNORE" in updated_cell_q4)

print("Test notebook updated:",
      test_notebook_q4.is_file())

print("Original project remains unchanged by this cell.")

print("STEP 10D.5: PREPARATION COMPLETE")


Table creation included: True
Data insertion included: True
Test notebook updated: True
Original project remains unchanged by this cell.
STEP 10D.5: PREPARATION COMPLETE


In [25]:

# Q4 Step 10D.6: Clean notebook execution test

import json
import tempfile
import subprocess
import sys
from pathlib import Path

test_folder_q4 = (
    Path.home() / "question_04_clean_test"
)

test_notebook_q4 = (
    test_folder_q4
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Read the prepared test notebook
with test_notebook_q4.open("r", encoding="utf-8") as f:
    notebook_data_q4 = json.load(f)

# Keep only the first nine core cells
core_cells_q4 = notebook_data_q4["cells"][:9]

assert len(core_cells_q4) == 9

# Create a fresh, isolated execution folder
with tempfile.TemporaryDirectory(
    prefix="q4_repro_"
) as temp_dir:

    temp_path = Path(temp_dir)

    # Adjust database path in temporary notebook
    temp_code_cells = []

    for cell in core_cells_q4:
        cell_copy = json.loads(json.dumps(cell))

        if cell_copy.get("cell_type") == "code":
            source = "".join(cell_copy.get("source", []))

            source = source.replace(
                'Path.home()\n    / "question_04_clean_test"',
                'Path(r"{}")'.format(temp_dir)
            )

            cell_copy["source"] = source.splitlines(True)

        temp_code_cells.append(cell_copy)

    notebook_data_q4["cells"] = temp_code_cells

    clean_notebook_q4 = temp_path / "Q4_Clean_Test.ipynb"

    with clean_notebook_q4.open("w", encoding="utf-8") as f:
        json.dump(notebook_data_q4, f, indent=1)

    # Execute using a fresh Jupyter kernel
    command_q4 = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(clean_notebook_q4),
        "--output",
        "Q4_Clean_Test_Executed.ipynb"
    ]

    result_q4 = subprocess.run(
        command_q4,
        cwd=str(temp_path),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )

    print("Execution return code:", result_q4.returncode)

    if result_q4.returncode != 0:
        print("Execution error:")
        print(result_q4.stderr[-2500:])

    print(
        "STEP 10D.6:",
        "PASS" if result_q4.returncode == 0
        else "REVIEW REQUIRED"
    )


Execution return code: 1
Execution error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/beakerx/bin/jupyter-nbconvert", line 7, in <module>
    from nbconvert.nbconvertapp import main
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/nbconvertapp.py", line 140, in <module>
    class NbConvertApp(JupyterApp):
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/nbconvertapp.py", line 225, in NbConvertApp
    """.format(formats=get_export_names()))
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/exporters/base.py", line 141, in get_export_names
    e = get_exporter(exporter_name)(config=config)
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/exporters/base.py", line 102, in get_exporter
    if getattr(exporter(config=config), 'enabled', True):
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/exporters/templateexporter.py", line 325, in __init__
    su

In [26]:

# Q4 Step 10D.6A: Diagnose notebook execution failure

if "result_q4" in globals():

    print("Execution return code:", result_q4.returncode)

    print("\nLAST 5000 CHARACTERS OF ERROR:")
    print("-" * 65)

    error_text_q4 = result_q4.stderr

    if error_text_q4:
        print(error_text_q4[-5000:])
    else:
        print("No stderr was captured.")

    print("\nSTANDARD OUTPUT:")
    print(result_q4.stdout[-1000:])

    print("\nSTEP 10D.6A: DIAGNOSTIC COMPLETE")

else:
    print("result_q4 is unavailable in this kernel.")
    print("Do not rerun the clean test yet.")
    print("STEP 10D.6A: PREVIOUS RESULT NOT AVAILABLE")


Execution return code: 1

LAST 5000 CHARACTERS OF ERROR:
-----------------------------------------------------------------
Traceback (most recent call last):
  File "/opt/anaconda3/envs/beakerx/bin/jupyter-nbconvert", line 7, in <module>
    from nbconvert.nbconvertapp import main
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/nbconvertapp.py", line 140, in <module>
    class NbConvertApp(JupyterApp):
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/nbconvertapp.py", line 225, in NbConvertApp
    """.format(formats=get_export_names()))
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/exporters/base.py", line 141, in get_export_names
    e = get_exporter(exporter_name)(config=config)
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/site-packages/nbconvert/exporters/base.py", line 102, in get_exporter
    if getattr(exporter(config=config), 'enabled', True):
  File "/opt/anaconda3/envs/beakerx/lib/python3.6/sit

In [28]:

# Q4 Step 10D.6B: Check notebook execution libraries

import sys
import importlib.util

print("Python version:", sys.version.split()[0])

packages_q4 = [
    "nbformat",
    "nbclient",
    "nbconvert",
    "jupyter_client",
    "ipykernel"
]

for package_q4 in packages_q4:

    try:
        available_q4 = (
            importlib.util.find_spec(package_q4)
            is not None
        )

        print(
            "{}: {}".format(
                package_q4,
                "AVAILABLE" if available_q4
                else "NOT AVAILABLE"
            )
        )

    except (ImportError, ValueError) as error_q4:
        print(
            "{}: CHECK REQUIRED ({})".format(
                package_q4, error_q4
            )
        )

print("\nSTEP 10D.6B: ENVIRONMENT CHECK COMPLETE")


Python version: 3.6.12
nbformat: AVAILABLE
nbclient: AVAILABLE
nbconvert: AVAILABLE
jupyter_client: AVAILABLE
ipykernel: AVAILABLE

STEP 10D.6B: ENVIRONMENT CHECK COMPLETE


In [29]:

# Q4 Step 10D.6C: Inspect nbclient execution API

import nbclient
import nbformat
import inspect

print("nbclient version:",
      getattr(nbclient, "__version__", "Unknown"))

print("nbformat version:",
      getattr(nbformat, "__version__", "Unknown"))

try:
    from nbclient import NotebookClient

    print("NotebookClient import: PASS")

    print(
        "NotebookClient signature:",
        inspect.signature(NotebookClient)
    )

    print(
        "Execute method available:",
        callable(
            getattr(NotebookClient, "execute", None)
        )
    )

    print("STEP 10D.6C: API CHECK COMPLETE")

except Exception as error_q4:

    print("NotebookClient import: FAILED")
    print("Error type:", type(error_q4).__name__)
    print("Error:", str(error_q4))

    print("STEP 10D.6C: REVIEW REQUIRED")


nbclient version: 0.5.3
nbformat version: 5.1.3
NotebookClient import: PASS
NotebookClient signature: (*args, **kwargs)
Execute method available: True
STEP 10D.6C: API CHECK COMPLETE


In [30]:

# Q4 Step 10D.7: Clean execution with nbclient

import json
import tempfile
from pathlib import Path

import nbformat
from nbclient import NotebookClient

# Locate the isolated notebook
test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

# Load notebook
with test_notebook_q4.open("r", encoding="utf-8") as f:
    notebook_json_q4 = json.load(f)

# Keep only core SQL cells 1-9
core_cells_q4 = notebook_json_q4["cells"][:9]

assert len(core_cells_q4) == 9
assert all(
    cell["cell_type"] == "code"
    for cell in core_cells_q4
)

# Create a fresh temporary directory
with tempfile.TemporaryDirectory(prefix="q4_clean_") as temp_dir:

    # Deep-copy the core cells
    core_copy_q4 = json.loads(
        json.dumps(core_cells_q4)
    )

    # Redirect the copied notebook's database path
    first_code_q4 = "".join(core_copy_q4[0]["source"])

    old_reference_q4 = (
        '/ "question_04_clean_test"'
    )

    new_reference_q4 = (
        '/ "{}"'.format(Path(temp_dir).name)
    )

    # Replace only the database directory expression
    # with an absolute temporary database path
    old_db_block_q4 = '''db_path_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "monthly_revenue.db"
)'''

    new_db_block_q4 = (
        'db_path_q4 = Path(r"{}")'.format(
            str(Path(temp_dir) / "monthly_revenue.db")
        )
    )

    assert first_code_q4.count(old_db_block_q4) == 1, (
        "Database setup block did not match."
    )

    first_code_q4 = first_code_q4.replace(
        old_db_block_q4,
        new_db_block_q4
    )

    core_copy_q4[0]["source"] = (
        first_code_q4.splitlines(True)
    )

    # Reject references to our working project
    all_core_code_q4 = "\n".join(
        "".join(cell["source"])
        for cell in core_copy_q4
    )

    assert "question_04_month_over_month" not in all_core_code_q4
    assert "question_04_clean_test" not in all_core_code_q4

    # Create an execution-ready notebook
    clean_nb_q4 = nbformat.from_dict(
        {
            "cells": core_copy_q4,
            "metadata": notebook_json_q4.get("metadata", {}),
            "nbformat": 4,
            "nbformat_minor": 4
        }
    )

    # Run cells in a fresh kernel
    try:
        client_q4 = NotebookClient(
            clean_nb_q4,
            timeout=120,
            kernel_name="python3",
            resources={
                "metadata": {
                    "path": temp_dir
                }
            }
        )

        executed_nb_q4 = client_q4.execute()

        # Confirm the temporary database was created
        test_db_path_q4 = (
            Path(temp_dir) / "monthly_revenue.db"
        )

        print(
            "Fresh test database created:",
            test_db_path_q4.is_file()
        )

        # Display final validation messages
        for cell in executed_nb_q4.cells:
            for output in cell.get("outputs", []):
                if output.get("output_type") == "stream":
                    output_text = "".join(
                        output.get("text", "")
                    )

                    if "VALIDATION" in output_text or "PASS" in output_text:
                        print(output_text.strip())

        print("STEP 10D.7: EXECUTION COMPLETED")

    except Exception as error_q4:
        print("Error type:", type(error_q4).__name__)
        print("Error details:")
        print(str(error_q4)[-3000:])
        print("STEP 10D.7: REVIEW REQUIRED")


Error type: AttributeError
Error details:
'list' object has no attribute 'strip'
STEP 10D.7: REVIEW REQUIRED


In [31]:

# Q4 Step 10D.7A: Diagnose notebook source types

import json
from pathlib import Path

test_notebook_q4 = (
    Path.home()
    / "question_04_clean_test"
    / "SQL_Question_04_Month_Over_Month_TEST.ipynb"
)

with test_notebook_q4.open(
    "r", encoding="utf-8"
) as f:
    notebook_json_q4 = json.load(f)

core_cells_q4 = notebook_json_q4["cells"][:9]

print("CORE NOTEBOOK CELL TYPES")
print("-" * 55)

for number_q4, cell_q4 in enumerate(core_cells_q4, start=1):

    source_q4 = cell_q4.get("source", "")

    print(
        "Cell {} | source type: {} | cell type: {}".format(
            number_q4,
            type(source_q4).__name__,
            cell_q4.get("cell_type")
        )
    )

print("-" * 55)
print("STEP 10D.7A: DIAGNOSTIC COMPLETE")


CORE NOTEBOOK CELL TYPES
-------------------------------------------------------
Cell 1 | source type: list | cell type: code
Cell 2 | source type: list | cell type: code
Cell 3 | source type: list | cell type: code
Cell 4 | source type: list | cell type: code
Cell 5 | source type: list | cell type: code
Cell 6 | source type: list | cell type: code
Cell 7 | source type: list | cell type: code
Cell 8 | source type: list | cell type: code
Cell 9 | source type: list | cell type: code
-------------------------------------------------------
STEP 10D.7A: DIAGNOSTIC COMPLETE
